# SU(6) determinant-sector preflight v2\nRun the single code cell. If the required contraction source is absent, Colab opens an upload picker. Upload **one** full symbolic bundle, or the two-file fallback shown by the cell.\n

In [ ]:
from pathlib import Path

SCRIPT = '#!/usr/bin/env python3\n"""\ny4_su6_determinant_stage0.py\n\nOne-block Colab/local preflight for the exceptional SU(6) O(y^4) band contraction.\n\nThis script does three things:\n  1. Certifies the exact mathematical reduction: at O(y^4), SU(6) differs from\n     the stable-rank N>=7 contraction only in final local Haar sectors (6,0)\n     and (0,6). All intermediate resolvent/Casimir data are unchanged.\n  2. Certifies the determinant Haar projector\n         int_SU(6) U^(tensor 6) dU = epsilon epsilon / 6!\n     and its signed-permutation realization.\n  3. Searches Colab, Drive, and /mnt/data for the full symbolic source bundle,\n     extracts it safely, inventories the source/word schemas, and identifies\n     source lines where the stable balanced-sector filter must be generalized.\n\nNo manual path edits are required. In Colab, if the contraction inputs are not\nfound, an explicit upload picker requests the exact source bundle. It does not\nclaim the SU(6) numerical band coefficient; it produces the exact preflight\nneeded before the finite-rank run.\n"""\n\nfrom __future__ import annotations\n\nimport ast\nimport gzip\nimport hashlib\nimport itertools\nimport json\nimport os\nimport re\nimport shutil\nimport sys\nimport time\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass\nfrom fractions import Fraction\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nVERSION = "2026-06-14-su6-determinant-stage0-v2"\nOUT_ROOT = Path("/content/SU6_DETERMINANT_STAGE0") if Path("/content").exists() else Path("/mnt/data/SU6_DETERMINANT_STAGE0")\nEXTRACT_ROOT = OUT_ROOT / "extracted"\nOUT_ROOT.mkdir(parents=True, exist_ok=True)\nEXTRACT_ROOT.mkdir(parents=True, exist_ok=True)\n\n\ndef gate(name: str, cond: bool, detail: str = "") -> None:\n    status = "PASS" if cond else "FAIL"\n    print(f"{status:4s} {name:68s} {detail}")\n    if not cond:\n        raise AssertionError(f"{name}: {detail}")\n\n\ndef sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for block in iter(lambda: f.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef perm_sign(p: tuple[int, ...]) -> int:\n    inv = sum(p[i] > p[j] for i in range(len(p)) for j in range(i + 1, len(p)))\n    return -1 if inv % 2 else 1\n\n\ndef certify_math() -> dict[str, Any]:\n    # A fourth-order matrix element has bra + ket + four perturbing plaquette\n    # characters: at most six U/Ubar factors on any one link.\n    max_final_local_degree = 2 + 4\n    max_resolvent_local_degree = 1 + 3  # ket plus at most three V\'s before a denominator\n\n    gate("O(y^4) final local tensor degree is at most six", max_final_local_degree == 6,\n         f"degree={max_final_local_degree}")\n    gate("all resolvent local tensor degrees are below six", max_resolvent_local_degree < 6,\n         f"degree<={max_resolvent_local_degree}")\n\n    # SU(N) selection rule: r-s = 0 mod N. Under r+s<=6 at N=6,\n    # the only non-balanced sectors are (6,0) and (0,6).\n    sectors = [(r, s) for r in range(7) for s in range(7-r) if (r-s) % 6 == 0]\n    exceptional = [(r, s) for (r, s) in sectors if r != s]\n    gate("SU(6) exceptional final sectors are exactly (6,0),(0,6)",\n         exceptional == [(0, 6), (6, 0)], str(exceptional))\n\n    resolvent_exceptional = [\n        (r, s) for r in range(max_resolvent_local_degree + 1)\n        for s in range(max_resolvent_local_degree + 1-r)\n        if r != s and (r-s) % 6 == 0\n    ]\n    gate("no determinant sector occurs in a fourth-order resolvent denominator",\n         not resolvent_exceptional, str(resolvent_exceptional))\n\n    perms = list(itertools.permutations(range(6)))\n    signs = [perm_sign(p) for p in perms]\n    sign_counts = Counter(signs)\n    gate("S6 determinant expansion has 720 terms", len(perms) == 720, str(len(perms)))\n    gate("S6 signs split 360 even / 360 odd", sign_counts == Counter({1: 360, -1: 360}), str(sign_counts))\n\n    coeff = Fraction(1, 720)\n    # Rank-one projector in the permutation basis:\n    # P_{sigma,tau}=sgn(sigma)sgn(tau)/720.\n    # Exact idempotence reduces to 720/720^2=1/720.\n    idempotent_coeff = sum(Fraction(s*s, 720*720) for s in signs)\n    gate("determinant projector normalization", idempotent_coeff == coeff, str(idempotent_coeff))\n\n    # Antisymmetry under adjacent transpositions.\n    def compose(a: tuple[int, ...], b: tuple[int, ...]) -> tuple[int, ...]:\n        return tuple(a[b[i]] for i in range(6))\n\n    adjacent = []\n    ident = tuple(range(6))\n    for i in range(5):\n        t = list(ident)\n        t[i], t[i+1] = t[i+1], t[i]\n        adjacent.append(tuple(t))\n    anti_ok = all(perm_sign(compose(t, p)) == -perm_sign(p) for t in adjacent for p in perms)\n    gate("determinant tensor is antisymmetric in every adjacent pair", anti_ok)\n\n    return {\n        "max_final_local_degree": max_final_local_degree,\n        "max_resolvent_local_degree": max_resolvent_local_degree,\n        "su6_admissible_sectors_r_plus_s_le_6": sectors,\n        "su6_exceptional_sectors": exceptional,\n        "determinant_projector": {\n            "formula": "Integral_SU(6) prod[a=1..6] U[i_a,j_a] dU = epsilon(i_1...i_6) epsilon(j_1...j_6)/720",\n            "delta_expansion": "epsilon(i)epsilon(j)=sum_{sigma in S6} sgn(sigma) prod_a delta(i_a,j_{sigma(a)})",\n            "coefficient": "1/720",\n            "permutation_terms": 720,\n            "even_terms": 360,\n            "odd_terms": 360,\n            "projector_idempotent": True,\n        },\n        "consequence": (\n            "SU(6) uses the stable-rank fourth-order denominators and fusion/Casimir data unchanged; "\n            "only final trace-network contractions containing local signatures (6,0) or (0,6) must be added."\n        ),\n    }\n\n\nKNOWN_ARCHIVE_PATTERNS = [\n    "Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE_2026-06-14*.zip",\n    "GLUEBALL_FLAT_BAND_SOURCE_RELEASE_V0_7*.zip",\n    "y4_extracted_sources*.zip",\n    "SU_N_STAGE3G_WIRING_BUNDLE*.zip",\n    "Y4_SUN_SYMBOLIC_QAB_COMPACT_BUNDLE*.zip",\n]\nKNOWN_DATA_PATTERNS = [\n    "y4_sun_stable_ordered_words.json.gz",\n    "y4_sun_walled_brauer_fixed_rank.py",\n    "CERT_Y4_sun_walled_brauer_full_symbolic_certificate_2026-06-14.json",\n]\n\n\ndef candidate_roots() -> list[Path]:\n    roots = [Path("/content"), Path("/mnt/data")]\n    drive = Path("/content/drive/MyDrive")\n    if Path("/content").exists():\n        try:\n            from google.colab import drive as colab_drive  # type: ignore\n            if not drive.exists():\n                colab_drive.mount("/content/drive", force_remount=False)\n        except Exception:\n            pass\n    if drive.exists():\n        roots.append(drive)\n    # Deduplicate existing paths.\n    out = []\n    seen = set()\n    for r in roots:\n        try:\n            rr = r.resolve()\n        except Exception:\n            rr = r\n        if r.exists() and str(rr) not in seen:\n            out.append(r)\n            seen.add(str(rr))\n    return out\n\n\ndef discover_files(roots: Iterable[Path]) -> list[Path]:\n    found: dict[str, Path] = {}\n    for root in roots:\n        for pat in KNOWN_ARCHIVE_PATTERNS + KNOWN_DATA_PATTERNS:\n            try:\n                for p in root.rglob(pat):\n                    if p.is_file():\n                        found[str(p.resolve())] = p\n            except (PermissionError, OSError):\n                continue\n    return sorted(found.values(), key=lambda p: (p.name, str(p)))\n\n\ndef safe_extract_zip(zpath: Path, dest: Path) -> list[Path]:\n    extracted = []\n    dest.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(zpath) as zf:\n        for info in zf.infolist():\n            target = (dest / info.filename).resolve()\n            if not str(target).startswith(str(dest.resolve()) + os.sep) and target != dest.resolve():\n                raise ValueError(f"unsafe ZIP path: {info.filename}")\n        zf.extractall(dest)\n        extracted = [dest / i.filename for i in zf.infolist() if not i.is_dir()]\n    return extracted\n\n\ndef recursively_extract(initial_archives: list[Path], max_depth: int = 3) -> dict[str, Any]:\n    queue = [(p, 0) for p in initial_archives]\n    seen_hashes = set()\n    records = []\n    while queue:\n        zpath, depth = queue.pop(0)\n        try:\n            h = sha256(zpath)\n        except OSError:\n            continue\n        if h in seen_hashes or depth > max_depth:\n            continue\n        seen_hashes.add(h)\n        label = re.sub(r"[^A-Za-z0-9_.-]+", "_", zpath.stem)[:100]\n        dest = EXTRACT_ROOT / f"d{depth}_{label}_{h[:10]}"\n        try:\n            files = safe_extract_zip(zpath, dest)\n            records.append({\n                "archive": str(zpath), "sha256": h, "depth": depth,\n                "destination": str(dest), "file_count": len(files), "status": "extracted"\n            })\n            if depth < max_depth:\n                for p in files:\n                    if p.suffix.lower() == ".zip":\n                        queue.append((p, depth + 1))\n        except Exception as e:\n            records.append({"archive": str(zpath), "sha256": h, "depth": depth,\n                            "status": "error", "error": repr(e)})\n    return {"records": records, "unique_archives": len(seen_hashes)}\n\n\nSOURCE_TERMS = re.compile(r"walled|brauer|ordered.?words|balanced|signature|fusion.?path|trace.?topolog|casimir", re.I)\nPATCH_TERMS = re.compile(r"N\\s*[><=]+\\s*7|rank\\s*[><=]+\\s*7|balanced|r\\s*==\\s*s|p\\s*==\\s*q|charge|signature|imbalance", re.I)\n\n\ndef collect_source_files(search_roots: Iterable[Path]) -> list[Path]:\n    out = []\n    for root in search_roots:\n        if not root.exists():\n            continue\n        try:\n            for p in root.rglob("*.py"):\n                try:\n                    text = p.read_text(encoding="utf-8", errors="replace")\n                except OSError:\n                    continue\n                if SOURCE_TERMS.search(text) or p.name == "y4_sun_walled_brauer_fixed_rank.py":\n                    out.append(p)\n        except (PermissionError, OSError):\n            continue\n    # Unique by content hash/path.\n    uniq = {}\n    for p in out:\n        try:\n            uniq[(sha256(p), p.name)] = p\n        except OSError:\n            pass\n    return sorted(uniq.values(), key=lambda p: (p.name, str(p)))\n\n\ndef source_audit(paths: list[Path]) -> list[dict[str, Any]]:\n    recs = []\n    for p in paths:\n        text = p.read_text(encoding="utf-8", errors="replace")\n        lines = text.splitlines()\n        snippets = []\n        for i, line in enumerate(lines, start=1):\n            if PATCH_TERMS.search(line):\n                lo, hi = max(1, i-2), min(len(lines), i+2)\n                snippets.append({\n                    "line": i,\n                    "context": "\\n".join(f"{j:5d}: {lines[j-1]}" for j in range(lo, hi+1))\n                })\n                if len(snippets) >= 60:\n                    break\n        functions = []\n        try:\n            tree = ast.parse(text)\n            functions = [n.name for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]\n        except SyntaxError:\n            pass\n        recs.append({\n            "path": str(p), "sha256": sha256(p), "bytes": p.stat().st_size,\n            "functions": functions[:200], "candidate_patch_snippets": snippets,\n        })\n    return recs\n\n\ndef load_json_any(path: Path) -> Any:\n    if path.name.endswith(".gz"):\n        with gzip.open(path, "rt", encoding="utf-8") as f:\n            return json.load(f)\n    with path.open("r", encoding="utf-8") as f:\n        return json.load(f)\n\n\nPAIR_KEYSETS = [\n    ("r", "s"), ("p", "q"), ("fund", "antifund"), ("fundamental", "antifundamental"),\n    ("n_fund", "n_antifund"), ("nfund", "nantifund"), ("u", "ubar"),\n    ("plus", "minus"), ("forward", "backward"),\n]\n\n\ndef extract_signature_pairs(obj: Any, path: str = "$", key_hint: str = "") -> list[tuple[str, int, int]]:\n    out: list[tuple[str, int, int]] = []\n    if isinstance(obj, dict):\n        low = {str(k).lower(): k for k in obj}\n        for a, b in PAIR_KEYSETS:\n            if a in low and b in low:\n                va, vb = obj[low[a]], obj[low[b]]\n                if isinstance(va, int) and isinstance(vb, int) and 0 <= va <= 12 and 0 <= vb <= 12:\n                    out.append((path, va, vb))\n        for k, v in obj.items():\n            out.extend(extract_signature_pairs(v, f"{path}.{k}", str(k)))\n    elif isinstance(obj, list):\n        hint = key_hint.lower()\n        if any(t in hint for t in ("signature", "bidegree", "charge", "balance")):\n            for i, v in enumerate(obj):\n                if (isinstance(v, (list, tuple)) and len(v) == 2 and\n                        all(isinstance(x, int) for x in v) and all(0 <= x <= 12 for x in v)):\n                    out.append((f"{path}[{i}]", int(v[0]), int(v[1])))\n        for i, v in enumerate(obj):\n            out.extend(extract_signature_pairs(v, f"{path}[{i}]", key_hint))\n    return out\n\n\ndef schema_summary(obj: Any, depth: int = 0, max_depth: int = 4) -> Any:\n    if depth >= max_depth:\n        return type(obj).__name__\n    if isinstance(obj, dict):\n        return {str(k): schema_summary(v, depth+1, max_depth) for k, v in list(obj.items())[:30]}\n    if isinstance(obj, list):\n        return {"type": "list", "length": len(obj), "sample": schema_summary(obj[0], depth+1, max_depth) if obj else None}\n    return type(obj).__name__\n\n\ndef inspect_word_archives(search_roots: Iterable[Path]) -> list[dict[str, Any]]:\n    candidates = []\n    for root in search_roots:\n        try:\n            candidates.extend(root.rglob("*ordered*words*.json*"))\n            candidates.extend(root.rglob("y4_sun_stable_ordered_words.json.gz"))\n        except (PermissionError, OSError):\n            pass\n    uniq = {}\n    for p in candidates:\n        if p.is_file():\n            try:\n                uniq[sha256(p)] = p\n            except OSError:\n                pass\n    results = []\n    for p in sorted(uniq.values(), key=lambda x: str(x)):\n        rec: dict[str, Any] = {"path": str(p), "sha256": sha256(p), "bytes": p.stat().st_size}\n        try:\n            obj = load_json_any(p)\n            rec["schema"] = schema_summary(obj)\n            pairs = extract_signature_pairs(obj)\n            rec["explicit_signature_pairs_found"] = len(pairs)\n            rec["imbalance_histogram"] = dict(sorted(Counter(r-s for _, r, s in pairs).items()))\n            rec["exceptional_pair_examples"] = [x for x in pairs if abs(x[1]-x[2]) == 6][:50]\n            rec["has_explicit_su6_exceptional_pairs"] = any(abs(r-s) == 6 for _, r, s in pairs)\n            if isinstance(obj, list):\n                rec["top_level_records"] = len(obj)\n            elif isinstance(obj, dict):\n                for key in ("ordered_words", "words", "records", "data"):\n                    if key in obj and isinstance(obj[key], list):\n                        rec["top_level_records"] = len(obj[key])\n                        rec["record_container_key"] = key\n                        break\n        except Exception as e:\n            rec["error"] = repr(e)\n        results.append(rec)\n    return results\n\n\ndef explicit_colab_upload() -> list[Path]:\n    """Request the exact missing contraction inputs in Colab.\n\n    Preferred: upload the single full symbolic bundle. Fallback: upload the\n    fixed-rank source and stable ordered-word archive as two individual files.\n    """\n    if not Path("/content").exists():\n        return []\n    try:\n        from google.colab import files as colab_files  # type: ignore\n    except Exception:\n        return []\n\n    print("\\nREQUIRED INPUT — upload ONE of the following choices:")\n    print("  Preferred single file:")\n    print("    Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE_2026-06-14.zip")\n    print("  OR fallback pair:")\n    print("    y4_sun_walled_brauer_fixed_rank.py")\n    print("    y4_sun_stable_ordered_words.json.gz")\n    print("\\nThe compact Q/A/B bundle and independent-rerun bundle are not sufficient.")\n    uploaded = colab_files.upload()\n    saved: list[Path] = []\n    for raw_name, data in uploaded.items():\n        name = Path(raw_name).name\n        target = Path("/content") / name\n        target.write_bytes(data)\n        saved.append(target)\n        print(f"  saved {len(data):,} bytes -> {target}")\n    return saved\n\n\ndef active_script_path() -> Path | None:\n    """Return this script path when running as a file; None inside raw notebooks."""\n    try:\n        p = Path(__file__)  # type: ignore[name-defined]\n    except NameError:\n        return None\n    return p if p.is_file() else None\n\n\ndef main() -> None:\n    t0 = time.time()\n    print("=" * 100)\n    print("SU(6) EXCEPTIONAL-RANK O(y^4) DETERMINANT-SECTOR PREFLIGHT")\n    print("=" * 100)\n    math_cert = certify_math()\n\n    roots = candidate_roots()\n    print("\\nSearch roots:")\n    for r in roots:\n        print(" ", r)\n    discovered = discover_files(roots)\n    print(f"\\nDiscovered {len(discovered)} named source/data candidates")\n    for p in discovered:\n        print(f"  {sha256(p)[:12]}  {p}")\n\n    # A mounted Drive is not evidence that the required source exists there.\n    # If no contraction source/data candidate is present, request the exact\n    # files explicitly rather than silently searching unrelated Drive trees.\n    if not discovered:\n        uploaded_now = explicit_colab_upload()\n        if uploaded_now:\n            roots = candidate_roots()\n            discovered = discover_files(roots)\n            print(f"\\nAfter upload: discovered {len(discovered)} named source/data candidates")\n            for p in discovered:\n                print(f"  {sha256(p)[:12]}  {p}")\n\n    archives = [p for p in discovered if p.suffix.lower() == ".zip"]\n    extraction = recursively_extract(archives)\n    print(f"\\nExtracted {extraction[\'unique_archives\']} unique archives")\n\n    all_search_roots = roots + [EXTRACT_ROOT]\n    source_files = collect_source_files(all_search_roots)\n    sources = source_audit(source_files)\n    print(f"\\nAudited {len(sources)} relevant Python source files")\n    for s in sources:\n        print(f"  {Path(s[\'path\']).name}: {len(s[\'candidate_patch_snippets\'])} candidate stable-rank/filter sites")\n\n    word_archives = inspect_word_archives(all_search_roots)\n    print(f"\\nInspected {len(word_archives)} ordered-word archives")\n    for w in word_archives:\n        print(f"  {w[\'path\']}")\n        print(f"    records={w.get(\'top_level_records\')} explicit_pairs={w.get(\'explicit_signature_pairs_found\')} "\n              f"su6_exceptional={w.get(\'has_explicit_su6_exceptional_pairs\')}")\n\n    target_source = [s for s in sources if Path(s["path"]).name == "y4_sun_walled_brauer_fixed_rank.py"]\n    target_words = [w for w in word_archives if Path(w["path"]).name == "y4_sun_stable_ordered_words.json.gz"]\n    source_ready = bool(target_source and target_words)\n\n    status = "SOURCE_READY" if source_ready else "SOURCE_INPUT_MISSING"\n    if source_ready:\n        next_action = (\n            "Patch the word-admissibility filter from r=s to (r-s) mod 6=0; split stable and determinant "\n            "sectors; evaluate every (6,0)/(0,6) local Haar node with the signed S6 determinant projector."\n        )\n    else:\n        next_action = (\n            "Upload Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE_2026-06-14.zip. "\n            "Fallback: upload both y4_sun_walled_brauer_fixed_rank.py and "\n            "y4_sun_stable_ordered_words.json.gz. The independent rerun and compact Q/A/B "\n            "bundles contain results, not the contraction source chain."\n        )\n\n    result = {\n        "meta": {"version": VERSION, "created": time.strftime("%Y-%m-%d %H:%M:%S"), "runtime_seconds": time.time()-t0},\n        "status": status,\n        "math_certificate": math_cert,\n        "search_roots": [str(x) for x in roots],\n        "discovered_files": [{"path": str(p), "sha256": sha256(p), "bytes": p.stat().st_size} for p in discovered],\n        "archive_extraction": extraction,\n        "source_audit": sources,\n        "ordered_word_audit": word_archives,\n        "required_target_source_found": bool(target_source),\n        "required_stable_words_found": bool(target_words),\n        "next_action": next_action,\n    }\n\n    json_path = OUT_ROOT / "SU6_DETERMINANT_STAGE0_REPORT.json"\n    json_path.write_text(json.dumps(result, indent=2, sort_keys=True), encoding="utf-8")\n\n    md_lines = [\n        "# SU(6) exceptional-rank fourth-order determinant-sector preflight",\n        "",\n        f"**Version:** `{VERSION}`  ",\n        f"**Status:** `{status}`",\n        "",\n        "## Exact reduction",\n        "",\n        "A fourth-order matrix element contains the ket plaquette, four perturbing plaquette characters,",\n        "and the bra plaquette. Hence a fixed link carries at most six fundamental/antifundamental factors:",\n        "",\n        r"\\[r_\\ell+s_\\ell\\le 6.\\]",\n        "",\n        r"For Haar integration over \\(SU(6)\\), the selection rule is \\(r_\\ell-s_\\ell\\equiv0\\pmod 6\\).",\n        "Within the degree-six bound, the only sectors absent at stable rank are",\n        "",\n        r"\\[(r_\\ell,s_\\ell)=(6,0),(0,6).\\]",\n        "",\n        "The three resolvent denominators occur after at most three perturbations, so their local degree is",\n        "at most four. Consequently no determinant sector enters an intermediate energy or Casimir:",\n        "the complete stable-rank fusion and denominator data remain valid for SU(6).",\n        "",\n        "## Local SU(6) replacement",\n        "",\n        r"\\[",\n        r"\\int_{SU(6)}\\prod_{a=1}^{6}U_{i_a j_a}\\,dU",\n        r"=\\frac{1}{6!}\\epsilon_{i_1\\cdots i_6}\\epsilon_{j_1\\cdots j_6}",\n        r"=\\frac1{720}\\sum_{\\sigma\\in S_6}\\operatorname{sgn}(\\sigma)",\n        r"\\prod_{a=1}^{6}\\delta_{i_a,j_{\\sigma(a)}}.",\n        r"\\]",\n        "",\n        "The conjugate formula handles `(0,6)`. The projector has 720 signed terms, split into 360 even",\n        "and 360 odd permutations, and its exact normalization/idempotence gates pass.",\n        "",\n        "## Computational status",\n        "",\n        f"- Target fixed-rank source found: **{bool(target_source)}**",\n        f"- Stable ordered-word archive found: **{bool(target_words)}**",\n        f"- Relevant Python files audited: **{len(sources)}**",\n        f"- Ordered-word archives inspected: **{len(word_archives)}**",\n        "",\n        "## Next action",\n        "",\n        next_action,\n        "",\n        "The required code change is localized: generalize final word admissibility from exact balance",\n        "to balance modulo six, retain all stable-sector contractions unchanged, and add the determinant",\n        "projector only at exceptional final Haar nodes.",\n    ]\n    md_path = OUT_ROOT / "SU6_DETERMINANT_STAGE0_REPORT.md"\n    md_path.write_text("\\n".join(md_lines) + "\\n", encoding="utf-8")\n\n    zip_path = OUT_ROOT.parent / "SU6_DETERMINANT_STAGE0_V2_BUNDLE.zip"\n    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:\n        script_path = active_script_path()\n        if script_path is not None:\n            zf.write(script_path, arcname=script_path.name)\n        zf.write(json_path, arcname=json_path.name)\n        zf.write(md_path, arcname=md_path.name)\n\n    print("\\n" + "=" * 100)\n    print("PREFLIGHT STATUS:", status)\n    print(next_action)\n    print("JSON:", json_path)\n    print("MD:  ", md_path)\n    print("ZIP: ", zip_path)\n    print("=" * 100)\n\n\nif __name__ == "__main__":\n    main()\n'
script_path = Path('/content/ENGINE_Y4_su6_determinant_stage0_v2.py')
script_path.write_text(SCRIPT, encoding='utf-8')
ns = {'__name__': '__main__', '__file__': str(script_path)}
exec(compile(SCRIPT, str(script_path), 'exec'), ns)
